<a href="https://colab.research.google.com/github/tsopronyuk/dfa-lexical-analyzer/blob/main/notebooks/02_numbers_dfa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================================
# LEXICAL ANALYZER AND COMPACT DFA VISUALIZER
# Topic: Numeric Constants (Binary, Octal, Decimal, Hexadecimal),
#        Identifiers, and Noise Filtering
# ============================================================================

from graphviz import Digraph
from IPython.display import display, Markdown, HTML

# ============================================================================
# 1. TRANSITION MATRIX AND CHARACTER CLASSIFICATION
# ============================================================================

# Character classes (columns):
# 0:'0', 1:'1', 2:'2-7', 3:'8-9', 4:'b/B', 5:'x/X', 6:'o/O', 7:'a-f', 8:'g-z/_', 9:OTHER

M = [
    #  0   1   2   3   4   5   6   7   8   9
    [  1,  2,  2,  2,  3,  3,  3,  3,  3, -6 ], # 0: Initial state
    [  2,  2,  2,  2,  4,  5,  6, -3, -3, -3 ], # 1: Read '0'
    [  2,  2,  2,  2, -3, -3, -3, -3, -3, -3 ], # 2: Decimal number [0-9]+
    [  3,  3,  3,  3,  3,  3,  3,  3,  3, -1 ], # 3: Identifier [a-zA-Z_][a-zA-Z0-9_]*
    [  7,  7, -3, -3, -3, -3, -3, -3, -3, -3 ], # 4: Read '0b'
    [  8,  8,  8,  8,  8, -3, -3,  8, -3, -3 ], # 5: Read '0x'
    [  9,  9,  9, -3, -3, -3, -3, -3, -3, -3 ], # 6: Read '0o'
    [  7,  7, -2, -2, -2, -2, -2, -2, -2, -2 ], # 7: Binary number [0-1]+
    [  8,  8,  8,  8,  8, -5, -5,  8, -5, -5 ], # 8: Hexadecimal number [0-9a-fA-F]+
    [  9,  9,  9, -4, -4, -4, -4, -4, -4, -4 ]  # 9: Octal number [0-7]+
]

# Descriptions of final states using two-line labels (\n) to reduce circle diameter
TOKEN_INFO = {
    -1: ("Identifier", "Ident\n[a-z_]", "#d4e6f1"),
    -2: ("Binary Constant", "Bin\nConst", "#d5f5e3"),
    -3: ("Decimal Constant", "Dec\nConst", "#fcf3cf"),
    -4: ("Octal Constant", "Oct\nConst", "#ebdef0"),
    -5: ("Hexadecimal Constant", "Hex\nConst", "#fdebd0"),
    -6: ("Skip", "Noise", "#fadbd8")
}

def class_symb(c):
    if c == '0': return 0
    if c == '1': return 1
    if c in '234567': return 2
    if c in '89': return 3
    if c in 'bB': return 4
    if c in 'xX': return 5
    if c in 'oO': return 6
    if c.lower() in 'abcdef': return 7
    if c.isalpha() or c == '_': return 8
    return 9 # OTHER (whitespaces, delimiters, operators)

In [2]:
# ============================================================================
# 2. LEXICAL ANALYZER DRIVER (WITH BACKTRACKING AND DYNAMIC SLICING)
# ============================================================================

def lexical_analyzer(text, verbose=False):
    tokens = []
    text_buffer = text + '\0' # End-of-stream marker

    start = 0
    i = 0
    state = 0

    if verbose:
        print(f"\n--- STEP-BY-STEP TEXT ANALYSIS: '{text}' ---")

    while start < len(text_buffer) - 1:
        c = text_buffer[i]
        cl = class_symb(c)
        next_state = M[state][cl]

        if next_state < 0:
            if next_state == -6:
                # 1. NOISE: Skip 1 character
                if verbose:
                    print(f"| [NOISE] Skipped character '{text_buffer[start]}' (pos={start})")
                start += 1
                i = start
                state = 0

            elif next_state == -3 and state in (4, 5, 6):
                # 2. INVALID PREFIX BACKTRACK (e.g. '0b2'):
                # Character '0' is recognized as a standalone Decimal Constant
                val = "0"
                tokens.append({"type": TOKEN_INFO[-3][0], "value": val})
                if verbose:
                    print(f"| [PREFIX BACKTRACK] '0' recognized as Decimal Constant")

                # Move anchor to the prefix character ('b', 'x', or 'o')
                start = start + 1
                i = start
                state = 0

            else:
                # 3. VALID TOKEN (VARIABLE LENGTH): Slice text_buffer[start:i]
                val = text_buffer[start:i]
                token_type = TOKEN_INFO[next_state][0]
                tokens.append({"type": token_type, "value": val})

                if verbose:
                    print(f"| [FOUND] {token_type}: '{val}' (pos {start}:{i})")

                # Pointer 'i' stopped at the character that broke the token.
                # Start the next lexeme analysis from this character!
                start = i
                state = 0
        else:
            state = next_state
            i += 1

    return tokens

In [3]:
# ============================================================================
# 3. OPTIMIZED COMPACT DFA VISUALIZER (GRAPHVIZ)
# ============================================================================

def visualize_dfa_compact():
    dot = Digraph('DFA_Numbers_ID_Compact', format='svg')

    # General layout settings
    dot.attr(rankdir='LR', size='10,4', ratio='compress')

    # Compact dimensions for standard nodes
    dot.attr('node', fontname='Helvetica', fontsize='9', height='0.25', width='0.25', margin='0.02')
    dot.attr('edge', fontname='Helvetica', fontsize='8')

    # Start node
    dot.node('start_node', shape='point', width='0')
    dot.edge('start_node', '0')

    # Internal states (0..9)
    for state in range(10):
        dot.node(str(state), shape='circle', style='filled', fillcolor='#f8f9fa')

    # Final states (small double circles thanks to \n and 8pt font)
    for neg, (t_type, label, color) in TOKEN_INFO.items():
        if neg == -6: continue # Noise is described in the caption note

        node_label = f"{neg}\n{label}"
        dot.node(
            str(neg),
            label=node_label,
            shape='doublecircle',
            style='filled',
            fillcolor=color,
            fontsize='8'
        )

    class_names = {0:'0', 1:'1', 2:'2-7', 3:'8-9', 4:'b/B', 5:'x/X', 6:'o/O', 7:'a-f', 8:'g-z/_', 9:'other'}

    # Add combined edges
    for state in range(10):
        grouped = {}
        for cls in range(10):
            nxt = M[state][cls]
            if nxt == -6: continue # Ignore noise
            grouped.setdefault(nxt, []).append(class_names[cls])

        for target, labels in grouped.items():
            lbl = ", ".join(labels) if len(labels) < 6 else "else"
            is_final = target < 0
            dot.edge(
                str(state),
                str(target),
                label=f" {lbl} ",
                style='dashed' if is_final else 'solid',
                color='#777777' if is_final else '#111111'
            )

    dot.attr(
        label='\nNote: All undescribed transitions automatically lead to state -6 (Noise / Skip)',
        fontsize='9',
        fontcolor='#555555'
    )

    return dot

In [4]:
# ============================================================================
# 4. MAIN DEMO BLOCK
# ============================================================================

print("🖼️ Generating compact DFA graph...")
dot_compact = visualize_dfa_compact()
display(HTML(f"<div style='background: white; padding: 15px; border-radius: 8px; text-align: center;'>{dot_compact.pipe().decode('utf-8')}</div>"))

# Test string: contains valid numbers, identifiers, noise, and invalid prefix (0b2_test)
sample_text = "0b1010var_10x1A3F0o755123450b2_test"
print(f"\n👉 Input text: \"{sample_text}\"")

# Step-by-step analysis with verbose logging
tokens = lexical_analyzer(sample_text, verbose=True)

# Output table
print("\n" + "=" * 60)
print(f"{'No.':>3} | {'Token Type':<25} | {'Value':<15}")
print("-" * 60)
for idx, t in enumerate(tokens):
    print(f"{idx+1:>3} | {t['type']:<25} | {t['value']:<15}")
print("=" * 60)

🖼️ Generating compact DFA graph...



👉 Input text: "0b1010var_10x1A3F0o755123450b2_test"

--- STEP-BY-STEP TEXT ANALYSIS: '0b1010var_10x1A3F0o755123450b2_test' ---
| [FOUND] Binary Constant: '0b1010' (pos 0:6)
| [FOUND] Identifier: 'var_10x1A3F0o755123450b2_test' (pos 6:35)

No. | Token Type                | Value          
------------------------------------------------------------
  1 | Binary Constant           | 0b1010         
  2 | Identifier                | var_10x1A3F0o755123450b2_test
